# 01 — Secrets Smoke Test
Verifies your API keys without printing secrets.
Writes `outputs/01_secrets_smoketest/manifest.json`.

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import requests

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def mask(s: str, keep=4):
    if not s: return ""
    s = str(s)
    return s[:keep] + "…" + "*"*6 if len(s) > keep else "*"*len(s)

def env(name: str, default=""):
    v = os.getenv(name, default)
    return v.strip() if isinstance(v, str) else v

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": [],
    }

def add_artifact(manifest: dict, path: Path, kind: str, meta=None):
    meta = meta or {}
    manifest["artifacts"].append({
        "kind": kind,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "sha256": sha256_file(path) if path.exists() and path.is_file() else None,
        "meta": meta,
    })

print("PROJECT_ROOT:", PROJECT_ROOT)
print("UTC now:", utcnow())

In [ ]:
import yaml
step = "01_secrets_smoketest"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)
manifest = manifest_base(step, config_paths=[CONFIGS/"run.yml"])

expected = [
    "CDSE_USERNAME","CDSE_PASSWORD",
    "CDSE_ID","CDSE_SECRET",
    "OPENWEATHER_KEY",
    "WAQI_TOKEN",
    "NEWS_API_KEY",
    "NEWS_DATA_IO_KEY",
    "GEMINI_API_KEY","GEMINI_MODEL",
    "MET_OFFICE_KEY","METOFFICE_API_KEY",
    "IQ_AIR_QUALITY_KEY",
    "GETAMBEE_AIR_QUALITY_KEY",
]

present = {k: bool(env(k)) for k in expected}
write_json(out/"secrets_presence.json", present)
add_artifact(manifest, out/"secrets_presence.json", "secrets_presence")

print("Secrets present:", sum(present.values()), "/", len(expected))
missing = [k for k,v in present.items() if not v]
if missing:
    print("Missing:", missing)

# CDSE password token
if env("CDSE_USERNAME") and env("CDSE_PASSWORD"):
    r = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={"client_id":"cdse-public","grant_type":"password","username":env("CDSE_USERNAME"),"password":env("CDSE_PASSWORD")},
        timeout=30
    )
    meta = {"ok": r.ok, "status": r.status_code}
    try:
        j = r.json()
        meta["json_keys"] = list(j.keys())
        meta["has_access_token"] = "access_token" in j
    except Exception:
        meta["json_keys"] = None
        meta["has_access_token"] = False
    write_json(out/"cdse_password_token_meta.json", meta)
    add_artifact(manifest, out/"cdse_password_token_meta.json", "http_meta")
    print("CDSE password token:", "OK" if meta["ok"] and meta["has_access_token"] else f"FAIL ({meta['status']})")
else:
    manifest["notes"].append("CDSE_USERNAME/CDSE_PASSWORD missing; skipped password token test")

# Sentinel Hub client-credentials token
if env("CDSE_ID") and env("CDSE_SECRET"):
    r = requests.post(
        "https://services.sentinel-hub.com/auth/realms/main/protocol/openid-connect/token",
        data={"grant_type":"client_credentials","client_id":env("CDSE_ID"),"client_secret":env("CDSE_SECRET")},
        timeout=30
    )
    meta = {"ok": r.ok, "status": r.status_code}
    try:
        j = r.json()
        meta["json_keys"] = list(j.keys())
        meta["has_access_token"] = "access_token" in j
    except Exception:
        meta["json_keys"] = None
        meta["has_access_token"] = False
    write_json(out/"sentinelhub_clientcred_meta.json", meta)
    add_artifact(manifest, out/"sentinelhub_clientcred_meta.json", "http_meta")
    print("Sentinel Hub client-credentials:", "OK" if meta["ok"] and meta["has_access_token"] else f"FAIL ({meta['status']})")
else:
    manifest["notes"].append("CDSE_ID/CDSE_SECRET missing; skipped client-credentials test")

# OpenWeather
if env("OPENWEATHER_KEY"):
    r = requests.get("https://api.openweathermap.org/data/2.5/weather",
                     params={"lat":51.5074,"lon":-0.1278,"appid":env("OPENWEATHER_KEY")},
                     timeout=30)
    write_json(out/"openweather_meta.json", {"ok": r.ok, "status": r.status_code})
    add_artifact(manifest, out/"openweather_meta.json", "http_meta")
    print("OpenWeather:", "OK" if r.ok else f"FAIL ({r.status_code})")

# WAQI
if env("WAQI_TOKEN"):
    r = requests.get("https://api.waqi.info/feed/geo:51.5074;0.1278/", params={"token": env("WAQI_TOKEN")}, timeout=30)
    j = r.json() if r.headers.get("content-type","").startswith("application/json") else {}
    write_json(out/"waqi_meta.json", {"ok": r.ok, "status": r.status_code, "waqi_status": j.get("status")})
    add_artifact(manifest, out/"waqi_meta.json", "http_meta")
    print("WAQI:", "OK" if (r.ok and j.get("status")=="ok") else "FAIL")

# NewsAPI
if env("NEWS_API_KEY"):
    r = requests.get("https://newsapi.org/v2/everything",
                     params={"q":"air quality UK","pageSize":1,"sortBy":"publishedAt","language":"en"},
                     headers={"X-Api-Key": env("NEWS_API_KEY")},
                     timeout=30)
    write_json(out/"newsapi_meta.json", {"ok": r.ok, "status": r.status_code})
    add_artifact(manifest, out/"newsapi_meta.json", "http_meta")
    print("NewsAPI:", "OK" if r.ok else f"FAIL ({r.status_code})")

# NewsData.io
if env("NEWS_DATA_IO_KEY"):
    r = requests.get("https://newsdata.io/api/1/news",
                     params={"apikey": env("NEWS_DATA_IO_KEY"), "q":"air quality UK","language":"en","size":1},
                     timeout=30)
    write_json(out/"newsdataio_meta.json", {"ok": r.ok, "status": r.status_code})
    add_artifact(manifest, out/"newsdataio_meta.json", "http_meta")
    print("NewsData.io:", "OK" if r.ok else f"FAIL ({r.status_code})")

# Gemini: list models
if env("GEMINI_API_KEY"):
    r = requests.get("https://generativelanguage.googleapis.com/v1/models",
                     params={"key": env("GEMINI_API_KEY")}, timeout=30)
    write_json(out/"gemini_models_meta.json", {"ok": r.ok, "status": r.status_code})
    add_artifact(manifest, out/"gemini_models_meta.json", "http_meta")
    print("Gemini models:", "OK" if r.ok else f"FAIL ({r.status_code})")
    if env("GEMINI_MODEL"):
        manifest["notes"].append("GEMINI_MODEL set (not printed).")

# IQAir
if env("IQ_AIR_QUALITY_KEY"):
    r = requests.get("https://api.airvisual.com/v2/nearest_city",
                     params={"lat":51.5074,"lon":-0.1278,"key": env("IQ_AIR_QUALITY_KEY")},
                     timeout=30)
    j = r.json() if r.headers.get("content-type","").startswith("application/json") else {}
    write_json(out/"iqair_meta.json", {"ok": r.ok, "status": r.status_code, "api_status": j.get("status")})
    add_artifact(manifest, out/"iqair_meta.json", "http_meta")
    print("IQAir:", "OK" if (r.ok and j.get("status")=="success") else f"FAIL ({r.status_code})")

# Ambee
if env("GETAMBEE_AIR_QUALITY_KEY"):
    r = requests.get("https://api.ambeedata.com/latest/by-lat-lng",
                     params={"lat":50.873,"lng":0.008},
                     headers={"x-api-key": env("GETAMBEE_AIR_QUALITY_KEY")},
                     timeout=30)
    write_json(out/"ambee_meta.json", {"ok": r.ok, "status": r.status_code})
    add_artifact(manifest, out/"ambee_meta.json", "http_meta")
    print("Ambee:", "OK" if r.ok else f"FAIL ({r.status_code})")

# Met Office presence only
if env("MET_OFFICE_KEY") or env("METOFFICE_API_KEY"):
    manifest["notes"].append("Met Office key present. DataHub endpoint validation will be done in notebook 05.")
else:
    manifest["notes"].append("No Met Office key found.")

write_json(out/"manifest.json", manifest)
print("Wrote:", out/"manifest.json")